In [1]:
!pip install timm -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
import numpy as np
from collections import Counter
import os
import time
import json
import shutil
import threading
import warnings
from PIL import ImageFile


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3,
        saturation=0.3, hue=0.1
    ),
    transforms.RandomRotation(20),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Imports and transforms ready!")

Imports and transforms ready!


In [ ]:
data_dir = '/kaggle/input/datasets/huzzi05/food-classification/Data Final/Classification Data'

subdirs = [d for d in os.listdir(data_dir)
           if os.path.isdir(os.path.join(data_dir, d))]
print(f"Data directory: {data_dir}")
print(f"Total classes: {len(subdirs)}")

total = 0
for cls in subdirs:
    cls_path = os.path.join(data_dir, cls)
    total += len(os.listdir(cls_path))
print(f"Total images: {total:,}")

Data directory: /kaggle/input/datasets/huzzi05/food-classification/Data Final/Classification Data
Total classes: 748


Total images: 685,084


In [5]:
full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
num_classes = len(full_dataset.classes)

print(f"Classes: {num_classes}")
print(f"Total images: {len(full_dataset):,}")
print(f"Sample classes: {full_dataset.classes[:5]}")

Classes: 748
Total images: 685,084
Sample classes: ['Abalone', 'Aburaage', 'Agedashi_tofu', 'Aligot', 'Almond_biscuit']


In [ ]:
total = len(full_dataset)
val_size = int(0.2 * total)
train_size = total - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

val_dataset.dataset.transform = val_transform


targets = [full_dataset.targets[i] for i in train_dataset.indices]
class_counts = Counter(targets)
class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    sampler=sampler,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print(f"Train: {train_size:,} | Val: {val_size:,}")
print(f"Batches per epoch: {len(train_loader):,}")
print("Dataloaders ready!")

Train: 548,068 | Val: 137,016
Batches per epoch: 4,282
Dataloaders ready!


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = timm.create_model(
    'efficientnet_b0',
    pretrained=True,
    num_classes=num_classes
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded!")
print(f"Parameters: {total_params:,}")
print(f"Classes: {num_classes}")

Using device: cuda


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model loaded!
Parameters: 4,965,736
Classes: 748


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=25,
    eta_min=1e-6
)

print("Loss, optimizer, scheduler ready!")

Loss, optimizer, scheduler ready!


In [ ]:
def auto_save_thread():
    while True:
        time.sleep(300)
        try:
            if os.path.exists('/kaggle/working/best_model.pth'):
                shutil.copy(
                    '/kaggle/working/best_model.pth',
                    '/kaggle/working/best_model_backup.pth'
                )
                print(f"Auto-backup at {time.strftime('%H:%M:%S')}")
        except:
            pass

t = threading.Thread(target=auto_save_thread, daemon=True)
t.start()
print("Auto-save thread started!")

Auto-save thread started!


In [ ]:
best_val_acc = 0.0
num_epochs = 20
SAVE_EVERY = 5

history = {
    'train_acc': [], 'val_acc': [],
    'train_loss': [], 'val_loss': []
}

print(f"Starting training for {num_epochs} epochs...")
print(f"Checkpoint saved every {SAVE_EVERY} epochs")
print("="*60)

for epoch in range(num_epochs):
    epoch_start = time.time()

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

        if batch_idx % 500 == 0:
            batch_acc = (preds == labels).float().mean().item()
            print(f"  Epoch {epoch+1}/{num_epochs} | "
                  f"Batch {batch_idx}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Acc: {batch_acc:.4f}")

    train_acc = train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    epoch_time = time.time() - epoch_start

    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{num_epochs} | Time: {epoch_time/60:.1f} mins")
    print(f"Train Loss: {avg_train_loss:.4f} | "
          f"Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | "
          f"Val Acc:   {val_acc*100:.2f}%")
    print(f"{'='*60}\n")

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'num_classes': num_classes,
            'classes': full_dataset.classes,
            'history': history,
        }, '/kaggle/working/best_model.pth')
        print(f"New best saved! Val Acc: {val_acc*100:.2f}%")

    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc,
            'best_val_acc': best_val_acc,
            'num_classes': num_classes,
            'classes': full_dataset.classes,
            'history': history,
        }, f'/kaggle/working/checkpoint_epoch_{epoch+1}.pth')
        print(f"Checkpoint saved: checkpoint_epoch_{epoch+1}.pth")

        with open('/kaggle/working/training_history.json', 'w') as f:
            json.dump(history, f, indent=2)
        print(f"History saved!")

print(f"\nTraining Complete!")
print(f"Best Validation Accuracy: {best_val_acc*100:.2f}%")

Starting training for 25 epochs...
Checkpoint saved every 5 epochs


  Epoch 1/25 | Batch 0/4282 | Loss: 6.6472 | Acc: 0.0000


  Epoch 1/25 | Batch 500/4282 | Loss: 3.3926 | Acc: 0.3828


  Epoch 1/25 | Batch 1000/4282 | Loss: 2.9598 | Acc: 0.5625


  Epoch 1/25 | Batch 1500/4282 | Loss: 3.3261 | Acc: 0.4219


...

  Epoch 4/25 | Batch 4000/4282 | Loss: 1.8813 | Acc: 0.8047


Auto-backup at 10:25:59


Auto-backup at 10:30:59


Auto-backup at 10:35:59



Epoch 4/25 | Time: 62.1 mins
Train Loss: 1.9233 | Train Acc: 76.09%
Val Loss:   2.5029 | Val Acc:   60.32%

